In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

base = '/content/drive/MyDrive/Analytics for Society/Analytics for Society Award/'
processed = base + 'data/processed/'

# Load clean datasets
df_student = pd.read_csv(processed + 'pisa_student_all_years.csv')
df_school  = pd.read_csv(processed + 'pisa_school_all_years.csv')
df_interventions = pd.read_csv(processed + 'intervention_library.csv')

df_student['YEAR'] = df_student['YEAR'].astype(str)
df_school['YEAR']  = df_school['YEAR'].astype(str)

print(f"Student: {len(df_student):,} rows")
print(f"School:  {len(df_school):,} rows")
print(f"Interventions: {len(df_interventions)} entries")

Mounted at /content/drive
Student: 734,122 rows
School:  98,220 rows
Interventions: 24 entries


In [2]:
# ── COMPONENT 1 — Equity gap metric ──────────────────────────────────────────

# Clean ESCS sentinel values
df_student['ESCS'] = df_student['ESCS'].replace([9999, -9999, 9999.0, -9999.0], np.nan)
df_student.loc[df_student['ESCS'].abs() > 10, 'ESCS'] = np.nan

def calculate_equity_gap(df):
    results = []
    for (cnt, year), group in df.groupby(['CNT', 'YEAR']):
        if len(group) < 10:
            continue
        q25 = group['ESCS'].quantile(0.25)
        q75 = group['ESCS'].quantile(0.75)
        bottom = group[group['ESCS'] <= q25]['MATH'].mean()
        top    = group[group['ESCS'] >= q75]['MATH'].mean()
        if pd.isna(bottom) or pd.isna(top):
            continue
        results.append({
            'CNT':         cnt,
            'YEAR':        year,
            'GAP':         round(top - bottom, 2),
            'AVG_MATH':    round(group['MATH'].mean(), 2),
            'AVG_READ':    round(group['READ'].mean(), 2),
            'AVG_SCIENCE': round(group['SCIENCE'].mean(), 2),
            'AVG_ESCS':    round(group['ESCS'].mean(), 4),
            'N_STUDENTS':  len(group)
        })
    return pd.DataFrame(results)

df_gap = calculate_equity_gap(df_student)
df_gap = df_gap.dropna(subset=['GAP'])

print(f"Equity gap dataset: {len(df_gap)} country-year combinations")
print(f"Global average gap: {df_gap['GAP'].mean():.1f} pts")
print(f"Countries: {df_gap['CNT'].nunique()}")
print(f"\nSample:")
print(df_gap.head(10).to_string())

Equity gap dataset: 368 country-year combinations
Global average gap: 81.8 pts
Countries: 107

Sample:
   CNT  YEAR     GAP  AVG_MATH  AVG_READ  AVG_SCIENCE  AVG_ESCS  N_STUDENTS
0  ALB  2009   66.99    385.08    394.84       399.87   -0.7762        2000
1  ALB  2018   44.64    441.70    412.35       423.54   -0.7227        2000
2  ALB  2022   52.40    376.94    366.34       383.89   -0.6047        2000
3  ARE  2009   79.10    430.95    439.55       446.43    0.2723        2000
4  ARE  2012   69.35    431.90    436.94       443.51    0.3252        2000
5  ARE  2015   47.32    421.01    426.17       430.22    0.5444        2000
6  ARE  2018   79.23    426.54    419.29       423.59    0.2264        2000
7  ARE  2022   55.20    421.87    403.20       422.75    0.3016        2000
8  ARG  2009  104.07    400.54    413.74       416.05   -0.4820        2000
9  ARG  2012   73.70    411.01    424.51       428.06   -0.4257        2000


In [3]:
# ── COMPONENT 2 — Equity risk score (0-100) ───────────────────────────────────

# Step 1 — Calculate country trajectory
df_gap_09 = df_gap[df_gap['YEAR'] == '2009'][['CNT', 'GAP']].rename(columns={'GAP': 'GAP_2009'})
df_gap_22 = df_gap[df_gap['YEAR'] == '2022'][['CNT', 'GAP']].rename(columns={'GAP': 'GAP_2022'})
df_traj = df_gap_09.merge(df_gap_22, on='CNT')
df_traj['CHANGE'] = (df_traj['GAP_2022'] - df_traj['GAP_2009']).round(1)

def classify_trajectory(change):
    if change < -10:
        return 'Closing'
    elif change > 10:
        return 'Widening'
    else:
        return 'Stable'

df_traj['TRAJECTORY'] = df_traj['CHANGE'].apply(classify_trajectory)

# Trajectory score component — widening = higher risk
traj_score_map = {'Closing': -10, 'Stable': 0, 'Widening': 10}
df_traj['TRAJ_SCORE'] = df_traj['TRAJECTORY'].map(traj_score_map)

print("Trajectory breakdown:")
print(df_traj['TRAJECTORY'].value_counts().to_string())

# Step 2 — Calculate country average gap for benchmarking
country_avg_gap = df_gap[df_gap['YEAR'] == '2022'].groupby('CNT')['GAP'].mean().reset_index()
country_avg_gap.columns = ['CNT', 'COUNTRY_AVG_GAP']

# Step 3 — Join school data with gap and trajectory
df_school_clean = df_school.copy()
df_school_clean['SCHSIZE'] = df_school_clean['SCHSIZE'].replace([99999, 9999, 0], np.nan)
df_school_clean.loc[df_school_clean['SCHSIZE'] > 10000, 'SCHSIZE'] = np.nan
df_school_clean['STRATIO'] = df_school_clean['STRATIO'].replace([9999, 9999.0, 0], np.nan)
df_school_clean.loc[df_school_clean['STRATIO'] > 100, 'STRATIO'] = np.nan

# Get 2022 school data
df_school_2022 = df_school_clean[df_school_clean['YEAR'] == '2022'].copy()

# Merge with gap and trajectory
df_model = df_school_2022.merge(country_avg_gap, on='CNT', how='left')
df_model = df_model.merge(df_traj[['CNT', 'TRAJECTORY', 'TRAJ_SCORE', 'CHANGE']],
                           on='CNT', how='left')

print(f"\nModel dataset: {len(df_model):,} rows")
print(f"Columns: {list(df_model.columns)}")

# Step 4 — Calculate equity risk score components

# Component A: Gap size score (0-60)
# How does country gap compare to global average?
global_avg = df_gap['GAP'].mean()  # 80.7
global_std = df_gap['GAP'].std()   # 28.0

df_model['GAP_SCORE'] = df_model['COUNTRY_AVG_GAP'].apply(
    lambda x: min(60, max(0,
        30 + (x - global_avg) / global_std * 15
    )) if pd.notna(x) else 30
).round(1)

# Component B:
# Widening = higher risk
df_model['TRAJ_SCORE_SCALED'] = df_model['TRAJ_SCORE'].apply(
    lambda x: 10 + x if pd.notna(x) else 10
)

# Component C: School variable score (0-20)
# STRATIO — higher ratio slightly correlates with lower gap (from EDA r=-0.22)
# SCHTYPE — public schools have slightly higher gaps
df_model['SCHOOL_SCORE'] = 10  # default

# Adjust for school type
schtype_adjustment = {3.0: 2, 1.0: -2, 2.0: 0}
df_model['SCHOOL_SCORE'] += df_model['SCHTYPE'].map(schtype_adjustment).fillna(0)

# Adjust for STRATIO
stratio_median = df_model['STRATIO'].median()
df_model['SCHOOL_SCORE'] += df_model['STRATIO'].apply(
    lambda x: -1 if pd.notna(x) and x > stratio_median else 1
    if pd.notna(x) and x < stratio_median else 0
)

df_model['SCHOOL_SCORE'] = df_model['SCHOOL_SCORE'].clip(0, 20)

# Step 5 — Final equity risk score
df_model['EQUITY_RISK_SCORE'] = (
    df_model['GAP_SCORE'] +
    df_model['TRAJ_SCORE_SCALED'] +
    df_model['SCHOOL_SCORE']
).round(1).clip(0, 100)

print("\nEquity risk score summary:")
print(df_model['EQUITY_RISK_SCORE'].describe().round(2).to_string())

print("\nSample scores by country:")
sample = df_model.groupby('CNT').agg(
    AVG_RISK=('EQUITY_RISK_SCORE', 'mean'),
    TRAJECTORY=('TRAJECTORY', 'first'),
    AVG_GAP=('COUNTRY_AVG_GAP', 'first')
).reset_index().sort_values('AVG_RISK', ascending=False)

print(sample.head(10).to_string())
print("\n...")
print(sample.tail(10).to_string())

Trajectory breakdown:
TRAJECTORY
Stable      25
Widening    20
Closing     16

Model dataset: 21,629 rows
Columns: ['CNT', 'SCHOOLID', 'SCHSIZE', 'SCHTYPE', 'STRATIO', 'YEAR', 'COUNTRY_AVG_GAP', 'TRAJECTORY', 'TRAJ_SCORE', 'CHANGE']

Equity risk score summary:
count    21629.00
mean        51.45
std         19.73
min         11.90
25%         36.60
50%         48.30
75%         63.80
max         93.00

Sample scores by country:
    CNT   AVG_RISK TRAJECTORY  AVG_GAP
14  CZE  91.767442   Widening   133.94
32  ISR  90.202591   Widening   123.76
70  SVK  90.185417   Widening   121.80
65  ROU  89.357252   Widening   121.00
10  CHE  86.738462   Widening   116.64
73  TAP  85.403297   Widening   115.93
5   BEL  82.661404   Widening   112.67
21  FRA  81.429078     Stable   125.10
41  LTU  79.967808   Widening   106.90
39  KOR  79.721505   Widening   107.52

...
    CNT   AVG_RISK TRAJECTORY  AVG_GAP
66  SAU  28.667876       None    49.78
25  GTM  26.613793       None    49.20
40  KSV  25.71921

In [4]:
# Debug — check what's happening with costs
df_int_test = df_interventions.copy()
df_int_test['total_cost'] = df_int_test['cost_per_pupil'] * 620

print("All interventions with total cost for 620 students:")
print(df_int_test[['intervention', 'cost_per_pupil', 'total_cost',
                    'evidence', 'impact_months']].to_string())

print(f"\nBudget: £10,000")
print(f"Interventions within budget: {len(df_int_test[df_int_test['total_cost'] <= 10000])}")
print(f"After evidence filter (>=2): {len(df_int_test[(df_int_test['total_cost'] <= 10000) & (df_int_test['evidence'] >= 2)])}")
print(f"After removing peer tutoring and homework: {len(df_int_test[(df_int_test['total_cost'] <= 10000) & (df_int_test['evidence'] >= 2) & (~df_int_test['intervention'].isin(['Peer tutoring', 'Homework']))])}")

All interventions with total cost for 620 students:
                        intervention  cost_per_pupil  total_cost  evidence  impact_months
0    Metacognition & self-regulation              40       24800         4              8
1   Reading comprehension strategies              40       24800         3              7
2                           Feedback              40       24800         4              6
3        Oral language interventions              40       24800         4              6
4                      Peer tutoring              40       24800         4              6
5             Collaborative learning              40       24800         2              5
6                           Homework              40       24800         1              5
7                   Mastery learning              40       24800         2              5
8                 One to one tuition             450      279000         3              5
9                            Phonics            

In [5]:
def get_interventions(
    country_code,
    annual_budget_gbp,
    school_size,
    existing_practices=None,
    disadvantaged_pct=0.25
):
    if existing_practices is None:
        existing_practices = []

    # Target group = disadvantaged students only
    target_students = round(school_size * disadvantaged_pct)

    df_int = df_interventions.copy()

    # Total cost applies only to target group
    df_int['target_students'] = target_students
    df_int['total_cost'] = df_int['cost_per_pupil'] * target_students

    # Filter out existing practices
    df_int = df_int[~df_int['intervention'].isin(existing_practices)]

    # Filter by budget
    df_int = df_int[df_int['total_cost'] <= annual_budget_gbp]

    # Filter positive impact and sufficient evidence
    df_int = df_int[df_int['impact_months'] > 0]
    df_int = df_int[df_int['evidence'] >= 2]

    # Sort by cost effectiveness
    df_int = df_int.sort_values('cost_effectiveness', ascending=False)
    df_int['rank'] = range(1, len(df_int) + 1)

    # Get country profile
    country_data = df_model[df_model['CNT'] == country_code]
    country_risk = country_data['EQUITY_RISK_SCORE'].mean()
    country_traj = country_data['TRAJECTORY'].iloc[0] \
        if len(country_data) > 0 else 'Unknown'
    country_gap  = country_data['COUNTRY_AVG_GAP'].iloc[0] \
        if len(country_data) > 0 else None

    return {
        'country_code':   country_code,
        'equity_risk':    round(country_risk, 1) if pd.notna(country_risk) else None,
        'trajectory':     country_traj,
        'country_gap':    round(country_gap, 1) if country_gap and pd.notna(country_gap) else None,
        'budget':         annual_budget_gbp,
        'school_size':    school_size,
        'target_students': target_students,
        'interventions':  df_int[['rank', 'intervention', 'gap_reduction_pts',
                                   'total_cost', 'cost_effectiveness',
                                   'evidence', 'impact_months']]
    }

# Test with Westbrook Academy
result = get_interventions(
    country_code='GBR',
    annual_budget_gbp=10000,
    school_size=620,
    existing_practices=['Peer tutoring', 'Homework']
)

print("=" * 60)
print(f"EquiTrack Report — {result['country_code']}")
print("=" * 60)
print(f"Equity risk score:    {result['equity_risk']} / 100")
print(f"Country trajectory:   {result['trajectory']}")
print(f"Country avg gap:      {result['country_gap']} pts")
print(f"Budget:               £{result['budget']:,}")
print(f"School size:          {result['school_size']} students")
print(f"Target group:         {result['target_students']} disadvantaged students")
print(f"\nRecommended interventions (within £{result['budget']:,} budget):")
print(result['interventions'].to_string(index=False))

EquiTrack Report — GBR
Equity risk score:    59.8 / 100
Country trajectory:   Stable
Country avg gap:      94.7 pts
Budget:               £10,000
School size:          620 students
Target group:         155 disadvantaged students

Recommended interventions (within £10,000 budget):
 rank                     intervention  gap_reduction_pts  total_cost  cost_effectiveness  evidence  impact_months
    1  Metacognition & self-regulation               28.0        6200               70.00         4              8
    2 Reading comprehension strategies               24.5        6200               61.25         3              7
    3                         Feedback               21.0        6200               52.50         4              6
    4      Oral language interventions               21.0        6200               52.50         4              6
    5           Collaborative learning               17.5        6200               43.75         2              5
    6                 Master

In [6]:
def calculate_realistic_reduction(interventions_df, current_gap,
                                   target_pct=0.25):
    """
    Calculate realistic gap reduction accounting for:
    1. Interventions apply to target group only (25% of students)
    2. Diminishing returns when stacking interventions
    3. Cap at 60% of current gap
    """
    sorted_ints = interventions_df.sort_values('gap_reduction_pts',
                                                ascending=False)

    total = 0
    for i, (_, row) in enumerate(sorted_ints.iterrows()):
        # Scale to target group only
        scaled_reduction = row['gap_reduction_pts'] * target_pct

        # Diminishing returns
        decay = 0.7 ** i
        contribution = scaled_reduction * decay
        total += contribution

    # Cap at 60% of current gap
    max_reduction = current_gap * 0.60
    realistic_total = min(total, max_reduction)

    return round(realistic_total, 1)

realistic_reduction = calculate_realistic_reduction(
    result['interventions'],
    result['country_gap'],
    target_pct=0.25
)

projected_gap = round(result['country_gap'] - realistic_reduction, 1)

print(f"Current country gap:               {result['country_gap']} pts")
print(f"Target group:                      25% of {result['school_size']} students = {result['target_students']}")
print(f"Realistic gap reduction:           {realistic_reduction} pts")
print(f"Projected gap after interventions: {projected_gap} pts")
print(f"Percentage improvement:            {round(realistic_reduction/result['country_gap']*100, 1)}%")

Current country gap:               94.7 pts
Target group:                      25% of 620 students = 155
Realistic gap reduction:           18.7 pts
Projected gap after interventions: 76.0 pts
Percentage improvement:            19.7%


In [8]:
# Save all model outputs
processed = '/content/drive/MyDrive/Analytics for Society/Analytics for Society Award/data/processed/'

# 1. Country trajectories
df_traj.to_csv(processed + 'country_trajectories.csv', index=False)
print("Saved: country_trajectories.csv")

# 2. Country gap metrics
df_gap.to_csv(processed + 'equity_gap_by_country_year.csv', index=False)
print("Saved: equity_gap_by_country_year.csv")

# 3. School risk scores
df_model[['CNT', 'SCHOOLID', 'SCHTYPE', 'SCHSIZE', 'STRATIO',
          'COUNTRY_AVG_GAP', 'TRAJECTORY', 'EQUITY_RISK_SCORE']].to_csv(
    processed + 'school_risk_scores.csv', index=False
)
print("Saved: school_risk_scores.csv")

# 4. Intervention library (already saved but confirm)
df_interventions.to_csv(processed + 'intervention_library.csv', index=False)
print("Saved: intervention_library.csv")

print("\nAll model outputs saved — ready for Streamlit app")
print(f"\nProcessed folder contents:")
import os
for f in sorted(os.listdir(processed)):
    print(f"  {f}")

Saved: country_trajectories.csv
Saved: equity_gap_by_country_year.csv
Saved: school_risk_scores.csv
Saved: intervention_library.csv

All model outputs saved — ready for Streamlit app

Processed folder contents:
  country_trajectories.csv
  equity_gap_by_country_year.csv
  intervention_library.csv
  pisa_school_all_years.csv
  pisa_student_2009_stratified.csv
  pisa_student_2012_stratified.csv
  pisa_student_2015_stratified.csv
  pisa_student_2018_stratified.csv
  pisa_student_2022_stratified.csv
  pisa_student_all_years.csv
  school_risk_scores.csv


In [9]:
import os

processed = '/content/drive/MyDrive/Analytics for Society/Analytics for Society Award/data/processed/'

files = [
    'school_risk_scores.csv',
    'intervention_library.csv',
    'country_trajectories.csv',
    'equity_gap_by_country_year.csv'
]

print("Model output file sizes:")
total = 0
for f in files:
    path = processed + f
    size_mb = os.path.getsize(path) / 1024 / 1024
    total += size_mb
    print(f"  {f:<40} {size_mb:.2f} MB")

print(f"\nTotal: {total:.2f} MB")
print(f"\nGitHub limits:")
print(f"  Single file warning: 50 MB")
print(f"  Single file hard limit: 100 MB")
print(f"  Recommended repo size: <1 GB")

Model output file sizes:
  school_risk_scores.csv                   0.99 MB
  intervention_library.csv                 0.00 MB
  country_trajectories.csv                 0.00 MB
  equity_gap_by_country_year.csv           0.02 MB

Total: 1.01 MB

GitHub limits:
  Single file warning: 50 MB
  Single file hard limit: 100 MB
  Recommended repo size: <1 GB
